# Cross-lingual ABSA Experiment Runner on Kaggle

This notebook contains the complete setup and execution scripts for running the Cross-lingual Aspect-Based Sentiment Analysis (ABSA) model training (S1, S2, S3) on Kaggle with GPU accelerator.

### Step 1: Copy Project to Writable Workspace & Set Directory
Since `/kaggle/input` is read-only, we dynamically scan and copy the project files to `/kaggle/working/project` to allow saving model checkpoints and evaluation results.

In [ ]:
import os
import shutil
from pathlib import Path

# 1. Destination directory (writable workspace)
dest_dir = Path("/kaggle/working/project")

# Clean previous directories
if dest_dir.exists():
    print(f"Removing old workspace: {dest_dir}")
    shutil.rmtree(dest_dir)

# 2. Dynamically scan /kaggle/input for project files
input_dir = Path("/kaggle/input")
found_root = None

print("Scanning /kaggle/input for project directory...")
for root, dirs, files in os.walk(input_dir):
    if "requirements.txt" in files and "config.yml" in files:
        found_root = Path(root)
        print(f"Found project root: {found_root}")
        break

if not found_root:
    for root, dirs, files in os.walk(input_dir):
        if "requirements.txt" in files or "config.yml" in files or "train.py" in files:
            found_root = Path(root)
            print(f"Found fallback project root: {found_root}")
            break

# 3. Copy to writeable directory
if found_root:
    print(f"Copying project files to {dest_dir}...")
    shutil.copytree(found_root, dest_dir, dirs_exist_ok=True)
    print("Copy completed successfully!")
else:
    print("ERROR: Could not find requirements.txt or config.yml.")
    print("Current structure of /kaggle/input:")
    for root, dirs, files in os.walk(input_dir):
        depth = root.replace(str(input_dir), "").count(os.sep)
        indent = "  " * depth
        print(f"{indent}{os.path.basename(root)}/")
        for f in files[:5]:
            print(f"{indent}  {f}")
    raise FileNotFoundError("Project root not found.")

# 4. Change working directory
%cd /kaggle/working/project
!ls -la

### Step 2: Dynamic Patching for `as_target_tokenizer` Depreciation
This cell fixes the `AttributeError: T5Tokenizer has no attribute as_target_tokenizer` error caused by newer transformers library versions installed on Kaggle.

In [ ]:
# Dynamically patch dataset.py to fix as_target_tokenizer depreciation error
import re
from pathlib import Path

dataset_file = Path("/kaggle/working/project/src/data/dataset.py")
if dataset_file.exists():
    with open(dataset_file, "r", encoding="utf-8") as f:
        code = f.read()

    old_str = """        with tokenizer.as_target_tokenizer():
            tgt_enc = tokenizer(
                targets, max_length=max_target_len, truncation=True,
                padding="max_length", return_tensors="pt",
            )"""
            
    new_str = """        tgt_enc = tokenizer(
            text_target=targets, max_length=max_target_len, truncation=True,
            padding="max_length", return_tensors="pt",
        )"""
        
    if old_str in code:
        code = code.replace(old_str, new_str)
        with open(dataset_file, "w", encoding="utf-8") as f:
            f.write(code)
        print("Patched src/data/dataset.py successfully!")
    elif "text_target=targets" in code:
        print("File was already patched!")
    else: 
        # Fallback Regex replacement
        code_sub, count = re.subn(
            r"with\s+tokenizer\.as_target_tokenizer\(\)\s*:\s*\n\s+tgt_enc\s*=\s*tokenizer\([\s\S]*?\)",
            r"tgt_enc = tokenizer(\n            text_target=targets, max_length=max_target_len, truncation=True,\n            padding=\"max_length\", return_tensors=\"pt\",\n        )",
            code
        )
        if count > 0:
            with open(dataset_file, "w", encoding="utf-8") as f:
                f.write(code_sub)
            print("Patched src/data/dataset.py via regex successfully!")
        else:
            print(" Code pattern to patch was not found in dataset.py.")
else:
    print(" File dataset.py not found in workspace!")

### Step 3: Install Dependencies & Check GPU status

In [ ]:
# Install dependencies
!pip install -r requirements.txt

# Check GPU
import torch
print("="*40)
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
print("="*40)

### Step 4: Prepare Data & Pre-download Models

In [ ]:
# 1. Prepare data
!python -m scripts.prepare_data --domains restaurant

# 2. Download models
!python scripts/download_models.py --output_dir ./pretrained_models

### Phần 1: Khánh Toàn (mT5 - Restaurant)
Chạy mô hình `mT5` trên tập dữ liệu `restaurant` cho cả 3 thiết lập S1, S2, S3.

In [ ]:
!python -m scripts.train --setting s1 --models mt5 --domains restaurant --targets vi de zh
!python -m scripts.train --setting s2 --models mt5 --domains restaurant --targets vi de zh --n_values 50 100 200 --seeds 42 123 456
!python -m scripts.train --setting s3 --models mt5 --domains restaurant --targets vi de zh --seeds 42 123 456

### Phần 2: Nguyễn Trọng Huỳnh An (mT5 - Phone)
Chạy mô hình `mT5` trên tập dữ liệu `phone` cho cả 3 thiết lập S1, S2, S3.

In [ ]:
!python -m scripts.train --setting s1 --models mt5 --domains phone --targets vi de zh
!python -m scripts.train --setting s2 --models mt5 --domains phone --targets vi de zh --n_values 50 100 200 --seeds 42 123 456
!python -m scripts.train --setting s3 --models mt5 --domains phone --targets vi de zh --seeds 42 123 456

### Phần 3: Nguyễn Anh Tấn (XLM-R & AG-CAN - Restaurant)
Chạy mô hình `XLM-R` và `AG-CAN` trên tập dữ liệu `restaurant` cho cả 3 thiết lập S1, S2, S3.

In [ ]:
!python -m scripts.train --setting s1 --models xlmr ag_can --domains restaurant --targets vi de zh
!python -m scripts.train --setting s2 --models xlmr ag_can --domains restaurant --targets vi de zh --n_values 50 100 200 --seeds 42 123 456
!python -m scripts.train --setting s3 --models xlmr ag_can --domains restaurant --targets vi de zh --seeds 42 123 456

### Phần 4: Trần Đạt (XLM-R & AG-CAN - Phone)
Chạy mô hình `XLM-R` và `AG-CAN` trên tập dữ liệu `phone` cho cả 3 thiết lập S1, S2, S3.

In [ ]:
!python -m scripts.train --setting s1 --models xlmr ag_can --domains phone --targets vi de zh
!python -m scripts.train --setting s2 --models xlmr ag_can --domains phone --targets vi de zh --n_values 50 100 200 --seeds 42 123 456
!python -m scripts.train --setting s3 --models xlmr ag_can --domains phone --targets vi de zh --seeds 42 123 456

### Phần 5: Tổng hợp Kết quả & Đóng gói (Leader)
Chạy script đánh giá để tạo biểu đồ và bảng CSV, sau đó nén toàn bộ kết quả để tải về.

In [ ]:
!python -m scripts.eval

!zip -r /kaggle/working/outputs.zip /kaggle/working/project/outputs/results /kaggle/working/project/outputs/figures